
# 04 Geopy: Turning Addresses Into Coordinates

Every notebook so far assumed you already had coordinates. In reality, land documents, surveys, and
buyer-submitted information usually describe a location as **text** an address, a neighborhood name,
a description like "behind the old market in Molyko." Geopy is the library that converts between that
text and real coordinates.

Geopy is **not** built on Shapely it's a standalone library, and it's the one library in this whole
curriculum that isn't really part of the "GeoPandas ecosystem" in a technical sense. It just happens to
be the tool you reach for at the very start of a pipeline, before anything else applies.

## Two directions

- **Forward geocoding**: address/text → coordinates. `"Molyko, Buea, Cameroon"` → `(9.2915, 4.1502)`
- **Reverse geocoding**: coordinates → address/text. `(9.2915, 4.1502)` → `"Molyko, Buea, Southwest Region, Cameroon"`

## An important practical note before you run this

Geopy's most common geocoder, **Nominatim**, is a free service backed by OpenStreetMap data which
means it needs a live internet connection to work, and it enforces a strict rate limit (roughly one
request per second) as a condition of free use. **The cells below may not execute in a fully sandboxed
environment with no general internet access they will work on your own machine.** Each cell includes a
fallback so you can still see the expected behavior and complete the exercises either way.


In [9]:

from geopy.geocoders import Nominatim

# The user_agent string is required by Nominatim's usage policy always set something identifying
# your own application, never leave it as a generic default in production code.
geolocator = Nominatim(user_agent="land_banking_learning_curriculum")

try:
    location = geolocator.geocode("Molyko, Buea, Cameroon", timeout=5)
    if location:
        print("Address: ", location.address)
        print("Latitude:", location.latitude)
        print("Longitude:", location.longitude)
    else:
        print("No result found for that query.")
except Exception as e:
    print("Geocoding call failed likely no internet access in this environment.")
    print("On your own machine with internet access, this will return a real result.")
    print("Error detail:", e)


Address:  Molyko, Buea, Fako, Southwest, Cameroun
Latitude: 4.1591154
Longitude: 9.2805172



## Reverse geocoding


In [10]:

try:
    location = geolocator.reverse("4.1502, 9.2915", timeout=5)
    print(location.address if location else "No result found.")
except Exception as e:
    print("Reverse geocoding call failed likely no internet access in this environment.")
    print("Error detail:", e)


Bolifamba, Buea, Fako, Southwest, Cameroun



## Rate limiting required for any real batch of addresses

If you're geocoding a list of many parcel addresses (exactly what you'd do onboarding a land banking
company's existing paper inventory), calling Nominatim in a tight loop will get you blocked. Geopy ships
a built-in `RateLimiter` for exactly this.


In [11]:

from geopy.extra.rate_limiter import RateLimiter

# min_delay_seconds=1 respects Nominatim's usage policy of roughly one request per second
geocode_with_delay = RateLimiter(geolocator.geocode, min_delay_seconds=1)

addresses = [
    "Molyko, Buea, Cameroon",
    "Bonapriso, Douala, Cameroon",
    "Great Soppo, Buea, Cameroon",
]

results = []
for addr in addresses:
    try:
        loc = geocode_with_delay(addr, timeout=5)
        results.append({
            "address": addr,
            "lat": loc.latitude if loc else None,
            "lon": loc.longitude if loc else None,
        })
    except Exception:
        results.append({"address": addr, "lat": None, "lon": None})

for r in results:
    print(r)


{'address': 'Molyko, Buea, Cameroon', 'lat': 4.1591154, 'lon': 9.2805172}
{'address': 'Bonapriso, Douala, Cameroon', 'lat': 4.0256151, 'lon': 9.6930153}
{'address': 'Great Soppo, Buea, Cameroon', 'lat': 4.1535353, 'lon': 9.2563348}



## From geocoded results to a GeoDataFrame

Once you have lat/lon pairs, converting them into a proper GeoDataFrame (so you can join, filter, and
query them like everything else in this curriculum) takes one line.


In [12]:

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Using the results list from above if geocoding failed (no internet), we substitute known
# approximate coordinates so the rest of this notebook still runs end to end
fallback_coords = {
    "Molyko, Buea, Cameroon": (4.1502, 9.2915),
    "Bonapriso, Douala, Cameroon": (4.0290, 9.7180),
    "Great Soppo, Buea, Cameroon": (4.1590, 9.2610),
}

rows = []
for r in results:
    lat, lon = r["lat"], r["lon"]
    if lat is None:
        lat, lon = fallback_coords[r["address"]]
    rows.append({"address": r["address"], "geometry": Point(lon, lat)})

geocoded_gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")
geocoded_gdf


,address,geometry
0,"Molyko, Buea, Cameroon",POINT (9.28052 4.15912)
1,"Bonapriso, Douala, Cameroon",POINT (9.69302 4.02562)
2,"Great Soppo, Buea, Cameroon",POINT (9.25633 4.15354)



## An alternative for distance: `geopy.distance` vs. reprojecting

Notebook 03 showed you the "correct" way to measure real distance: reproject to a projected CRS, then
measure. Geopy offers a shortcut for the common case of **point-to-point distance directly from lat/lon**,
using a geodesic (curved-Earth) formula no manual reprojection needed. It's worth knowing both:

- **`geopy.distance.geodesic`** quick, accurate, point-to-point only. Great for "how far apart are
  these two addresses?"
- **Reprojecting with GeoPandas/PyProj (notebook 03)** needed for anything beyond simple point
  distance: area, buffers, polygon operations. Geopy's shortcut doesn't help you there.


In [13]:

from geopy.distance import geodesic

molyko = (4.1502, 9.2915)      # geopy takes (latitude, longitude) note the order is opposite to Shapely's (x, y)!
great_soppo = (4.1590, 9.2610)

distance_km = geodesic(molyko, great_soppo).km
print(f"Distance: {distance_km:.2f} km")


Distance: 3.52 km



**Order warning worth internalizing now:** Shapely/GeoPandas use `(x, y)` = `(longitude, latitude)`.
Geopy uses `(latitude, longitude)`. Mixing these up silently produces a point on the wrong side of the
planet with no error message this is one of the most common real bugs in geospatial code, and it's
worth deliberately double-checking every time you cross between these two libraries.

## Exercises

### Exercise 1
Write a function `geocode_or_fallback(address, fallback_dict, geolocator)` that tries to geocode an
address with the live geolocator, and falls back to a lookup in `fallback_dict` if geocoding fails or
returns nothing generalizing the pattern used above into something reusable.


In [14]:
# Your code here


#### Solution

In [15]:

def geocode_or_fallback(address, fallback_dict, geolocator):
    try:
        loc = geolocator.geocode(address, timeout=5)
        if loc:
            return (loc.latitude, loc.longitude)
    except Exception:
        pass
    return fallback_dict.get(address)

result = geocode_or_fallback("Molyko, Buea, Cameroon", fallback_coords, geolocator)
print(result)


(4.1591154, 9.2805172)



### Exercise 2
Using `geopy.distance.geodesic`, find which of the three addresses in `fallback_coords` is closest to a
given reference point `(4.15, 9.28)` (an arbitrary point roughly in central Buea).


In [16]:
# Your code here


#### Solution

In [17]:

reference = (4.15, 9.28)

distances = {
    addr: geodesic(reference, coords).km
    for addr, coords in fallback_coords.items()
}

closest = min(distances, key=distances.get)
print("Distances (km):", distances)
print("Closest:", closest)


Distances (km): {'Molyko, Buea, Cameroon': 1.2770312825125456, 'Bonapriso, Douala, Cameroon': 50.441593569039064, 'Great Soppo, Buea, Cameroon': 2.3325231998822265}
Closest: Molyko, Buea, Cameroon



## What's next

So far every geometry has lived in memory, in a single Python session. Real parcel data needs to persist
somewhere, and needs to stay fast to query even with tens of thousands of parcels. **First**,
`05_spatial_indexing_rtree.ipynb` covers *why* and *how* queries stay fast at scale. **Then**,
`07_postgis_integration_psycopg2.ipynb` covers actually persisting this data in the same Postgres
database your product's backend already uses.
